# 03 — Model Training & Selection

Baseline vs XGBoost, six-candidate deterministic grid (D-013). All numbers read
from `results/metrics/*.json`.

Reproduce: `python scripts/run_all.py` (stages 5–8).

In [ ]:
import json
b = json.load(open('../results/metrics/baselines_val.json'))
sel = json.load(open('../results/metrics/model_selection.json'))

rows = [('prior',    b['results']['prior']['roc_auc'],    b['results']['prior']['pr_auc'],    b['results']['prior']['brier']),
        ('logistic', b['results']['logistic']['roc_auc'], b['results']['logistic']['pr_auc'], b['results']['logistic']['brier']),
        ('iforest',  b['results']['iforest']['roc_auc'],  b['results']['iforest']['pr_auc'],  b['results']['iforest']['brier'])]
rows += [(c['model'], c['roc_auc'], c['pr_auc'], c['brier']) for c in sel['candidates']]
rows.sort(key=lambda r: -r[1])

print(f"{'model':<18}{'ROC-AUC':>9}{'PR-AUC':>9}{'Brier':>9}")
for m, ra, pa, br in rows:
    print(f"{m:<18}{ra:>9.4f}{pa:>9.4f}{br:>9.4f}")
print(f"\nselected: {sel['selected']} (rule: {sel['rule']})")

In [ ]:
# Family ablation — what each feature family contributes on its own
ab = json.load(open('../results/metrics/ablation_families_only.json'))
print(f"{'family':<13}{'n_feat':>7}{'ROC-AUC':>9}{'PR-AUC':>9}{'event_recall':>14}")
for r in sorted(ab['results'], key=lambda r: -r['pr_auc']):
    print(f"{r['subset']:<13}{r['n_features']:>7}{r['roc_auc']:>9.4f}{r['pr_auc']:>9.4f}{r['event_recall']:>14.4f}")

**Findings:** XGBoost dominates every baseline. The dynamics family (rolling
z-scores, momentum, CV) carries most of the signal alone; calendar-only is
near-useless (PR-AUC ≈ 0.06). Class weighting lifts event recall but wrecks
calibration (Brier 0.0234 vs 0.0064) — rejected, see `xgb_weighted_val.json`.